# LLM08: Mixture of Experts (MoE) & Numerical Precision

## Lab Overview

1. **Mixture of Experts (MoE)**: A sparsely-activated architecture that scales model capacity without proportionally increasing compute. Models like Mixtral 8×7B, DeepSeek-V3 (256+1 experts, top-8+1), and Qwen3-235B (128 experts, top-8) use MoE to activate only a subset of parameters per token.
2. **Numerical Precision & Quantization**: Understanding FP32, BF16, FP16, FP8 representations and how quantization (INT8/INT4) reduces model size and speeds up inference.

> **Quick term:** **MoE (Mixture of Experts)** means many separate "expert" feed-forward networks plus a **router** that picks which expert(s) run for each token—only a few experts activate, so compute stays manageable while capacity grows.

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

By the end of this lab, you will be able to:

1. Understand the MoE architecture: expert networks, gating mechanism, and sparse activation.
2. Implement Top-K gating and expert routing from scratch.
3. Explore load balancing strategies for MoE training.
4. Compare floating-point formats (FP32, FP16, BF16, FP8) and their trade-offs.
5. Implement basic quantization (INT8/INT4) and measure its impact on model accuracy.

---


## 1. Environment Setup

In [ ]:
import math
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)

## 2. MoE Concept

### Why MoE Emerged

As language models scaled to hundreds of billions of parameters, researchers faced a fundamental challenge: **dense models don't scale efficiently**.

**The Scaling Dilemma:**
- Larger models achieve better quality but require proportionally more compute
- Training a 100B+ dense model is prohibitively expensive
- Inference becomes impractical: every token requires all parameters

**MoE Solution:** Decouple **model capacity** from **compute cost**
- Increase parameters by adding more experts (capacity grows)
- Keep compute constant by activating only $k$ experts per token
- Result: GPT-level quality with manageable inference cost

**Key Insight:** Not all knowledge needs to be activated for every token. A token like "python" should activate programming experts, while "parrot" activates biology experts.

###  MoE vs Dense

| Aspect | Dense Model FFN | MoE Model Expert |
|--------|-----------------|------------------|
| **Structure** | Single FFN (e.g., 4096→16384→4096) | Multiple smaller FFNs (e.g., 8× [4096→4096→4096]) |
| **Activation** | Always 100% active | Top-k active (e.g., 2 of 8 = 25%) |
| **Specialization** | General-purpose, averaged | Each expert learns different patterns |
| **Parameters** | All used every forward | Only subset used per token |

**In Dense Models:** The single FFN must learn to handle ALL types of tokens — grammar, facts, reasoning, code, etc. This forces it to be a "jack of all trades."

**In MoE Models:** Each expert can specialize:
- Expert 1: Code tokens
- Expert 2: Mathematical expressions
- Expert 3: Common grammar patterns
- Expert 4: Rare vocabulary

### Transformer Layer Structure: Where Does MoE Fit?

A standard Transformer layer consists of two main components:

```
Input
  │
  ▼
┌─────────────────────────┐
│  Multi-Head Attention   │  ← CANNOT be MoE (dense by nature)
│    (MHA Layer)          │
└─────────────────────────┘
  │
  ▼
┌─────────────────────────┐
│   Feed-Forward Network  │  ← CAN be replaced with MoE
│      (FFN Layer)        │
└─────────────────────────┘
  │
  ▼
Output
```

### Why MHA Layer Cannot Be MoE

The Multi-Head Attention (MHA) layer **cannot** be converted to MoE for fundamental reasons:

1. **Token Mixing Requirement:** MHA must compute attention between ALL token pairs. Each query needs to attend to all keys — this is inherently dense.

2. **Context Dependency:** The attention output for token $i$ depends on tokens $j$ for all $j$. You cannot split this computation across independent experts.

3. **Mathematical Structure:** 
   $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V$$
   The $QK^T$ operation creates an $N \times N$ attention matrix — every element matters.

4. **Information Flow:** Sparse activation would break the attention mechanism, as each token needs global context.

**Therefore:** MoE only replaces the FFN layer, not the MHA layer.

### Compute and Memory: Dense vs MoE (Same Model Size)

Let's compare a 8B parameter dense model vs a MoE model with the same total size:

**Configuration:**
- Dense: Single FFN with 8B total parameters
- MoE: 8 experts, each 1B, total 8B parameters, top-2 activation

| Metric | Dense 8B | MoE 8B (8 experts, top-2) |
|--------|----------|---------------------------|
| **Total Parameters** | 8B | 8B |
| **Active Parameters** | 8B (100%) | 2B (25%) |
| **Memory Footprint** | 32 GB (FP32) | 32 GB (FP32) |
| **Compute per Token** | 8B ops | ~2B ops (4× less!) |
| **Inference Speed** | 1× (baseline) | ~3-4× faster* |

*Speedup depends on implementation efficiency and memory bandwidth.



The Mixture of Experts idea is simple: instead of one large FFN, use **multiple smaller FFNs** ("experts") and a **gating network** ("router") that decides which expert(s) to activate for each input token.

**Key equation:**
$$y = \sum_{i=1}^{M} G(x)_i \cdot E_i(x)$$

where $G(x)$ is the gating function and $E_i$ is expert $i$.

**Sparsity via Top-K:**
$$G(x) = \text{softmax}(\text{TopK}(x \cdot W_g))$$

Only $k$ experts (typically $k=1$ or $2$) are activated per token. This means:
- **Model capacity** scales with $M$ (number of experts)
- **Compute cost** stays roughly at $k \times$ single-expert cost


### MoE Penalties vs Dense Models

Despite the compute savings, MoE introduces several penalties:

| Penalty | Description | Impact |
|---------|-------------|--------|
| **Communication Overhead** | In distributed training, experts may be on different GPUs, requiring all-to-all communication | 10-30% slowdown |
| **Load Imbalance** | Some experts may receive more tokens than others, causing GPU idle time | Wasted compute |
| **Router Compute** | Gating network adds extra computation per token | Small overhead (~5%) |

**Key Takeaway:** MoE is a powerful tool for scaling models, but it comes with its own set of challenges. 

Here are some extra cost for MoE than Dense Models. 




**Notable MoE models:**

| Model | #Experts | Top-K | Notes |
|-------|----------|-------|-------|
| Mixtral 8×7B | 8 | 2 | Popular open-source MoE |
| Mixtral 8×22B | 8 | 2 | Larger experts |
| DeepSeek-V3 (670B/37B active) | 256+1 | 8+1 | Shared expert + routing |
| Qwen3-235B-A22B | 128 | 8 | Alibaba open-source |

#### Gated Router Compute Cost

The router (gating network) computes:

$$G(x) = \text{softmax}(\text{TopK}(x \cdot W_g))$$

**Compute breakdown per token:**
1. **Linear projection:** $x \cdot W_g$ where $W_g \in \mathbb{R}^{d \times E}$
   - Cost: $O(d \cdot E)$ multiply-adds
2. **Top-K selection:** Find $k$ largest values among $E$
   - Cost: $O(E \log E)$ or $O(E)$ with optimized algorithms
3. **Softmax over k values:** Normalize selected scores
   - Cost: $O(k)$

**Total router cost:** $O(d \cdot E + E \log E)$ per token

For typical values ($d=4096$, $E=8$, $k=2$):
- Router: ~32K ops/token
- Expert (FFN): ~131M ops/token
- **Router overhead: ~0.02%** (negligible!)

In [ ]:
# TODO: Implement Top-K Gating Network
# Step 1: Implement the Top-K Gating Network

class TopKGating(nn.Module):
    """Router that selects top-k experts for each token.

    The gating network:
    1. Projects input to expert logits via linear layer
    2. Selects top-k experts per token
    3. Normalizes selected scores with softmax
    4. Creates sparse gate with only top-k nonzero values
    """

    def __init__(self, hidden_dim: int, num_experts: int, k: int = 2):
        super().__init__()
        self.router = nn.Linear(hidden_dim, num_experts, bias=False)
        self.k = k
        self.num_experts = num_experts

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: (batch, seq_len, hidden_dim)
        Returns:
            gate_probs: (batch, seq_len, num_experts) — sparse, only top-k nonzero
            topk_idx:   (batch, seq_len, k) — indices of selected experts
        """
        # TODO: Compute router scores (logits for each expert)
        scores = None  # Replace with correct code

        # TODO: Select top-k scores and their indices
        topk_scores, topk_idx = None, None  # Replace with correct code

        # TODO: Normalize selected scores with softmax
        probs = None  # Replace with correct code

        # TODO: Build sparse gate: only top-k positions are nonzero
        # Use scatter_ to place probabilities at selected indices
        gate = torch.zeros_like(scores)
        gate = None  # Replace with correct code using scatter_

        return gate, topk_idx


# Test the router
hidden_dim, num_experts, k = 64, 8, 2
gating = TopKGating(hidden_dim, num_experts, k).to(device)

x = torch.randn(2, 5, hidden_dim, device=device)  # batch=2, seq_len=5
gate, idx = gating(x)

print(f"Input shape: {x.shape}")
print(f"Gate shape:  {gate.shape}  (sparse — only {k} nonzero per token)")
print(f"Top-K idx:   {idx.shape}")
print(f"\nSample gate[0,0]: {gate[0, 0].detach().cpu().tolist()}")
print(f"Selected experts for token [0,0]: {idx[0, 0].tolist()}")
print(f"Gate sums to 1: {gate[0, 0].sum().item():.4f}")

In [ ]:
# Router Compute Cost Analysis
# Analyze the computational cost of the gating network

def analyze_router_compute(hidden_dim, num_experts, k, batch_size, seq_len):
    """Analyze compute cost of the router (gating network).

    Router computes: G(x) = softmax(TopK(x @ W_g))

    Args:
        hidden_dim: Input hidden dimension (d)
        num_experts: Number of experts (E)
        k: Top-k selection
        batch_size: Batch size (B)
        seq_len: Sequence length (S)

    Returns:
        dict with compute breakdown
    """
    total_tokens = batch_size * seq_len

    # Step 1: Linear projection x @ W_g
    # W_g shape: (hidden_dim, num_experts)
    # For each token: hidden_dim * num_experts multiply-adds
    linear_ops = total_tokens * hidden_dim * num_experts * 2  # multiply + add

    # Step 2: Top-K selection
    # Using naive sorting: O(E log E) per token
    # For small E, this is negligible
    import math
    topk_ops = total_tokens * num_experts * math.log2(num_experts) if num_experts > 1 else 0

    # Step 3: Softmax over k values
    # exp, sum, divide for each of k values
    softmax_ops = total_tokens * k * 3  # exp + partial sum + divide

    # Total router ops
    total_router_ops = linear_ops + topk_ops + softmax_ops

    # Compare with expert FFN cost
    # FFN: gate_proj + up_proj + down_proj
    intermediate_dim = hidden_dim * 4  # Typical ratio
    expert_ops = total_tokens * (hidden_dim * intermediate_dim * 2 * 2 + intermediate_dim * hidden_dim * 2)

    return {
        'linear_proj_ops': linear_ops,
        'topk_ops': topk_ops,
        'softmax_ops': softmax_ops,
        'total_router_ops': total_router_ops,
        'single_expert_ffn_ops': expert_ops,
        'router_overhead_pct': total_router_ops / expert_ops * 100
    }


# Analyze router compute for typical configuration
print("=== Router Compute Cost Analysis ===")
hidden_dim = 4096
num_experts = 8
k = 2
batch_size = 1
seq_len = 512

analysis = analyze_router_compute(hidden_dim, num_experts, k, batch_size, seq_len)

print(f"Configuration: d={hidden_dim}, E={num_experts}, k={k}, tokens={batch_size*seq_len}")
print(f"\nRouter compute breakdown:")
print(f"  Linear projection (x @ W_g): {analysis['linear_proj_ops']:>12,} ops")
print(f"  Top-K selection:             {analysis['topk_ops']:>12,.0f} ops")
print(f"  Softmax (k values):          {analysis['softmax_ops']:>12,} ops")
print(f"  ─────────────────────────────────────")
print(f"  Total router:                {analysis['total_router_ops']:>12,} ops")
print(f"\nSingle expert FFN cost:        {analysis['single_expert_ffn_ops']:>12,} ops")
print(f"\nRouter overhead: {analysis['router_overhead_pct']:.3f}% of single expert FFN")
print(f"\nNote: With top-{k} activation, we use 2 experts, so:")
active_expert_ops = k * analysis['single_expert_ffn_ops']
print(f"  Active expert compute: {active_expert_ops:,} ops")
print(f"  Router overhead: {analysis['total_router_ops']/active_expert_ops*100:.3f}% of active compute")

#### Combining Layer: Concentrating Expert Outputs

After experts process their assigned tokens, outputs must be combined. The combining operation is:

$$y = \sum_{i=1}^{M} G(x)_i \cdot E_i(x)$$

where $G(x)_i$ is the gate weight for expert $i$ and $E_i(x)$ is the expert output.

**Compute Cost:** For each token:
- $k$ multiplications (gate weight × expert output) per output dimension
- $(k-1)$ additions to sum the weighted outputs
- Total: $O(k \cdot d)$ per token, where $d$ is hidden dimension

This is negligible compared to expert computation ($O(d \cdot d_{intermediate})$).


In [ ]:
# TODO: Demonstrate Combining Layer - Concentrating Expert Outputs
# This shows how expert outputs are weighted and combined

def combine_expert_outputs(expert_outputs, gate_probs, topk_idx):
    """Combine expert outputs using gate weights.

    This is the key operation that merges results from multiple experts.
    Formula: output = Σᵢ gate_weight[i] × expert_output[i]

    Args:
        expert_outputs: List of tensors, each (batch, seq_len, hidden_dim)
        gate_probs: (batch, seq_len, num_experts) — sparse gate weights
        topk_idx: (batch, seq_len, k) — indices of selected experts

    Returns:
        combined_output: (batch, seq_len, hidden_dim)
        combine_ops: Number of multiply-add operations performed
    """
    B, S, D = expert_outputs[0].shape
    num_experts = len(expert_outputs)
    k = topk_idx.shape[-1]

    # Initialize output tensor
    output = torch.zeros(B, S, D, device=expert_outputs[0].device)

    # Count operations for analysis
    multiply_ops = 0
    add_ops = 0

    # TODO: Combine expert outputs weighted by gate probabilities
    for i in range(num_experts):
        # Get gate weight for this expert (B, S, 1)
        expert_weight = gate_probs[:, :, i].unsqueeze(-1)

        # Only compute if some tokens selected this expert
        if expert_weight.sum() > 0:
            # Multiply expert output by gate weight
            weighted_output = expert_outputs[i] * expert_weight

            # Accumulate to output
            output += weighted_output

            # Count operations: D multiplies + D adds per token
            multiply_ops += B * S * D
            add_ops += B * S * D

    total_ops = multiply_ops + add_ops
    return output, total_ops


# Demonstrate combining with concrete example
B, S, D = 2, 5, 64  # batch=2, seq_len=5, hidden=64
num_experts, k = 8, 2

# Simulate expert outputs (each expert produces different output)
expert_outputs = [torch.randn(B, S, D, device=device) for _ in range(num_experts)]

# Get gate probabilities from router
gate_probs, topk_idx = gating(x)

# Combine expert outputs
combined_output, ops_count = combine_expert_outputs(expert_outputs, gate_probs, topk_idx)

print("=== Combining Layer Demo ===")
print(f"Input: {num_experts} expert outputs, each shape: {expert_outputs[0].shape}")
print(f"Gate shape: {gate_probs.shape} (sparse, top-{k} per token)")
print(f"Combined output shape: {combined_output.shape}")
print(f"\nCombine operations count: {ops_count:,} ops")
print(f"  - Multiplies: {B*S*D*k:,} (k={k} experts × {B*S} tokens × {D} dims)")
print(f"  - Adds: {B*S*D*(k-1):,} (sum k values)")
print(f"\nCompute cost: O(k × B × S × D) = O({k} × {B} × {S} × {D})")
print(f"\nComparison with expert FFN cost:")
intermediate_dim = 256
expert_ffn_ops = B * S * (D * intermediate_dim * 2 + intermediate_dim * D)  # gate+up + down
combine_ops = B * S * D * k * 2  # multiply + add
print(f"  Expert FFN: {expert_ffn_ops:,} ops")
print(f"  Combine layer: {combine_ops:,} ops")
print(f"  Combine overhead: {combine_ops/expert_ffn_ops*100:.2f}%")

### LLaMA-Style FFN Expert: SwiGLU Activation

Before implementing the Expert module, let's recall the **LLaMA FFN structure** that each expert will use.

**Standard Transformer FFN** (pre-LLaMA):
```
FFN(x) = W₂ · ReLU(W₁ · x)
```

**LLaMA FFN with SwiGLU** (Swish Gated Linear Unit):
```
FFN(x) = W_down · (SiLU(W_gate · x) ⊗ W_up · x)
```

where `⊗` is element-wise multiplication.

**Key Components:**
| Layer | Shape | Purpose |
|-------|-------|---------|
| `gate_proj` | d → d_inter | Produces gate activations |
| `up_proj` | d → d_inter | Produces value activations |
| `down_proj` | d_inter → d | Projects back to hidden dim |
| `SiLU` | - | Smooth activation: SiLU(x) = x · σ(x) |


**Compute Cost per Token:**
- Gate projection: $O(d \cdot d_{inter})$
- Up projection: $O(d \cdot d_{inter})$
- Down projection: $O(d_{inter} \cdot d)$
- **Total**: $O(3 \cdot d \cdot d_{inter})$ per token

For typical values ($d=4096$, $d_{inter}=11008$): ~135M ops/token per expert.

In [ ]:
# TODO: Implement Expert FFN (LLaMA-style)
# Step 2: Implement individual Expert (standard FFN with SiLU gating, LLaMA-style)

class Expert(nn.Module):
    """A single FFN expert — identical structure to LLaMA's MLP.

    Uses SwiGLU activation: SiLU(gate) * up
    """

    def __init__(self, hidden_dim: int, intermediate_dim: int):
        super().__init__()
        # TODO: Initialize projection layers
        self.gate_proj = None  # Replace with nn.Linear
        self.up_proj   = None  # Replace with nn.Linear
        self.down_proj = None  # Replace with nn.Linear
        # Initialize activation function as SiLU
        self.act_fn    = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with SwiGLU activation.

        Formula: down_proj(SiLU(gate_proj(x)) * up_proj(x))
        """
        # TODO: Implement SwiGLU forward pass
        return None  # Replace with correct code


# Test a single expert
expert = Expert(hidden_dim=64, intermediate_dim=256).to(device)
out = expert(x)
print(f"Expert input: {x.shape} -> output: {out.shape}")
print(f"Expert parameters: {sum(p.numel() for p in expert.parameters()):,}")

In [ ]:
# TODO: Assemble the full MoE Layer
# Step 3: Assemble the full MoE Layer

class MoELayer(nn.Module):
    """Mixture of Experts layer that replaces the standard FFN.

    The MoE layer:
    1. Routes each token to top-k experts via gating
    2. Computes each expert's output
    3. Combines outputs weighted by gate probabilities
    """

    def __init__(self, hidden_dim: int, intermediate_dim: int,
                 num_experts: int = 8, top_k: int = 2):
        super().__init__()
        self.gate = TopKGating(hidden_dim, num_experts, top_k)
        self.experts = nn.ModuleList(
            [Expert(hidden_dim, intermediate_dim) for _ in range(num_experts)]
        )
        self.num_experts = num_experts
        self.top_k = top_k

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (batch, seq_len, hidden_dim)"""
        B, S, D = x.shape
        gate_probs, topk_idx = self.gate(x)  # (B,S,E), (B,S,k)

        # TODO: Initialize output tensor
        output = None  # Replace with correct code

        # TODO: Compute each expert's output weighted by gate probability
        for i, expert in enumerate(self.experts):
            # TODO: Get gate weight for this expert
            expert_weight = None  # Replace with correct code

            # TODO: Only compute if some tokens selected this expert
            if None:  # Replace with correct condition
                expert_out = expert(x)  # (B, S, D)
                # TODO: Accumulate weighted expert output
                output = None  # Replace with correct code

        return output


# Instantiate and test
moe = MoELayer(hidden_dim=64, intermediate_dim=256, num_experts=8, top_k=2).to(device)
moe_out = moe(x)

total_params = sum(p.numel() for p in moe.parameters())
single_expert_params = sum(p.numel() for p in moe.experts[0].parameters())
active_params = moe.top_k * single_expert_params + sum(p.numel() for p in moe.gate.parameters())

print(f"MoE output shape: {moe_out.shape}")
print(f"Total parameters:  {total_params:>10,}")
print(f"Active parameters: {active_params:>10,}  ({moe.top_k} experts + router)")
print(f"Sparsity ratio:    {active_params/total_params:.2%}")

## 3. Load Balancing in MoE Training

Without any constraint, the router tends to send most tokens to the same few "popular" experts, causing:
- Some experts are over-trained, others under-trained
- GPU load imbalance in distributed training

**Solution**: Add an auxiliary **load-balancing loss** that encourages uniform expert utilization:

$$\mathcal{L}_{\text{balance}} = \alpha \cdot N_E \cdot \sum_{i=1}^{N_E} f_i \cdot P_i$$

where $f_i$ is the fraction of tokens routed to expert $i$ and $P_i$ is the average router probability for expert $i$.

*Note: DeepSeek-V3 uses a different approach — shared experts with capacity constraints instead of auxiliary loss.*

In [ ]:
# TODO: Implement load balancing loss
def load_balancing_loss(gate_probs: torch.Tensor, topk_idx: torch.Tensor,
                        num_experts: int) -> torch.Tensor:
    """
    Compute Switch Transformer load-balancing loss.

    The loss encourages uniform expert utilization by penalizing:
    - Experts that receive too many tokens (high f_i)
    - Experts with high average probabilities (high P_i)

    Args:
        gate_probs: (B, S, E) — sparse gate probabilities
        topk_idx:   (B, S, k) — indices of selected experts
        num_experts: total number of experts

    Returns:
        loss: scalar loss value
    """
    B, S, E = gate_probs.shape
    total_tokens = B * S

    # TODO: f_i — fraction of tokens routed to expert i
    expert_counts = torch.zeros(E, device=gate_probs.device)
    for i in range(E):
        expert_counts[i] = None  # Count tokens routed to expert i
    f = None  # Normalize by total_tokens

    # TODO: P_i — average probability assigned to expert i
    P = None  # Mean over batch and sequence dimensions

    # TODO: Compute loss = N_E * sum(f_i * P_i)
    loss = None  # Replace with correct code
    return loss


# Demonstrate load balancing
gate, idx = gating(x)
lb_loss = load_balancing_loss(gate, idx, num_experts)

# Check expert distribution
print("=== Expert Load Distribution ===")
for i in range(num_experts):
    count = (idx == i).sum().item()
    avg_prob = gate[:, :, i].mean().item()
    print(f"  Expert {i}: {count:>3d} tokens, avg prob = {avg_prob:.4f}")
print(f"\nLoad-balancing loss: {lb_loss.item():.4f}")
print(f"Ideal (uniform): {1.0:.4f}")

## 4. Floating-Point Format Comparison

| Format | Sign | Exponent | Mantissa | Approx Range | Common Use |
|--------|:----:|:--------:|:--------:|:-------------|:-----------|
| FP32   | 1    | 8        | 23       | $\pm 3.4 \times 10^{38}$ | Full-precision training, critical evaluation |
| BF16   | 1    | 8        | 7        | $\pm 3.4 \times 10^{38}$ | Large-scale training (same range as FP32) |
| FP16   | 1    | 5        | 10       | $\pm 6.55 \times 10^{4}$ | Mixed-precision training & inference |
| FP8 E4M3 | 1 | 4        | 3        | $\pm 448$               | Latest GPU inference/training |
| FP8 E5M2 | 1 | 5        | 2        | $\pm 5.7 \times 10^{4}$ | Latest GPU training (wider range) |

**Trade-off**: More exponent bits → wider dynamic range; more mantissa bits → higher precision.

In [ ]:
# Compare precision across formats
print("=== Floating-Point Format Properties ===")

formats = {
    "FP32": torch.float32,
    "FP16": torch.float16,
    "BF16": torch.bfloat16,
}

x_fp32 = torch.tensor([0.1, 1.0, 100.0, 65504.0, 1e-8], dtype=torch.float32)

print(f"{'Format':>6s} | {'Size (bits)':>11s} | {'eps':>12s} | {'max':>12s} | {'Representation of 0.1':>22s}")
print("-" * 75)
for name, dtype in formats.items():
    info = torch.finfo(dtype)
    val = torch.tensor(0.1, dtype=dtype)
    print(f"{name:>6s} | {info.bits:>11d} | {info.eps:>12.2e} | {info.max:>12.2e} | {val.item():.20f}")

# Demonstrate precision loss
print("\n=== Precision Loss When Casting ===")
large_val = torch.tensor(100000.0, dtype=torch.float32)
for name, dtype in formats.items():
    casted = large_val.to(dtype)
    # Check for overflow
    if casted.isinf():
        print(f"{name}: 100000.0 -> inf (OVERFLOW)")
    else:
        error = abs(casted.float().item() - large_val.item())
        print(f"{name}: 100000.0 -> {casted.float().item()}, error = {error}")

## 5. Mixed Precision Training

Mixed precision combines FP32 (master weights, loss scaling) with FP16/BF16 (forward/backward computation) to:
- **Halve** memory usage
- **Double** throughput on Tensor Cores
- Maintain FP32-level training accuracy

Typical flow:
1. Forward pass in FP16/BF16
2. Loss computed in FP32 (with loss scaling for FP16)
3. Backward pass in FP16/BF16
4. Weight update in FP32 (master copy)

In [ ]:
# Demonstrate mixed precision with a simple MLP
class SimpleMLP(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * 4)
        self.fc2 = nn.Linear(dim * 4, dim)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

dim = 256
model_fp32 = SimpleMLP(dim).to(device)

# Memory comparison
def model_memory(model, dtype=None):
    """Calculate model memory footprint for a given dtype.

    Args:
        model: PyTorch model
        dtype: target dtype (or None for original)

    Returns:
        total bytes required for model parameters
    """
    total = 0
    for p in model.parameters():
        if dtype:
            # TODO: Calculate memory with target dtype
            total += None  # Replace with correct code
        else:
            # TODO: Calculate memory with original dtype
            total += None  # Replace with correct code
    return total

print("=== Memory Comparison ===")
for name, dtype in [("FP32", torch.float32), ("FP16", torch.float16), ("BF16", torch.bfloat16)]:
    mem = model_memory(model_fp32, dtype)
    print(f"{name}: {mem/1024:.1f} KB")

# Forward pass in different precisions
x_test = torch.randn(4, dim, device=device)

with torch.no_grad():
    out_fp32 = model_fp32(x_test)
    model_fp16 = SimpleMLP(dim).to(device).half()
    model_fp16.load_state_dict({k: v.half() for k, v in model_fp32.state_dict().items()})
    out_fp16 = model_fp16(x_test.half())

    error = (out_fp32 - out_fp16.float()).abs().mean().item()
    print(f"\nFP32 vs FP16 output mean absolute error: {error:.6f}")

## 6. Quantization Basics

Quantization maps floating-point weights to a lower-bit integer representation:

$$x_q = \text{round}\left(\frac{x}{\text{scale}} + \text{zero\_point}\right)$$

$$\text{scale} = \frac{x_{\max} - x_{\min}}{q_{\max} - q_{\min}}, \qquad
  \text{zero\_point} = \text{round}\left(q_{\min} - \frac{x_{\min}}{\text{scale}}\right)$$

**Common approaches:**
- **Post-Training Quantization (PTQ)**: Quantize after training with a small calibration set
- **Quantization-Aware Training (QAT)**: Insert fake-quantize nodes during training
- **Weight-only INT8/INT4**: Quantize only weights; activations stay in FP16

In [ ]:
# TODO: Implement quantization functions
def quantize_tensor(x: torch.Tensor, num_bits: int = 8):
    """Symmetric quantization of a float tensor to num_bits integers.

    Symmetric quantization uses:
    - qmin = -(2^(num_bits-1))
    - qmax = 2^(num_bits-1) - 1
    - scale = max(|x|) / qmax
    - x_q = round(x / scale), clipped to [qmin, qmax]

    Returns:
        x_q:   quantized integer tensor
        scale: float scale factor for dequantization
    """
    # TODO: Calculate quantization range
    qmin = None  # Replace with correct formula
    qmax = None  # Replace with correct formula

    # TODO: Calculate scale factor
    x_max = None  # Maximum absolute value
    scale = None  # x_max / qmax

    # TODO: Quantize and clip to range
    x_q = None  # round(x / scale), clipped to [qmin, qmax]
    return x_q, scale


def dequantize_tensor(x_q: torch.Tensor, scale: float) -> torch.Tensor:
    """Dequantize: int tensor -> float tensor.

    Formula: x_dequantized = x_q * scale
    """
    # TODO: Dequantize by multiplying with scale
    return None  # Replace with correct code


# Demo: quantize model weights
weight = model_fp32.fc1.weight.detach().cpu()
print(f"Original weight: shape={weight.shape}, dtype={weight.dtype}")
print(f"  range: [{weight.min():.4f}, {weight.max():.4f}]")
print(f"  memory: {weight.numel() * weight.element_size()} bytes")

# INT8 quantization
w_q8, scale8 = quantize_tensor(weight, num_bits=8)
w_deq8 = dequantize_tensor(w_q8, scale8)
error8 = (weight - w_deq8).abs().mean().item()

print(f"\nINT8 quantized: dtype={w_q8.dtype}")
print(f"  memory: {w_q8.numel() * w_q8.element_size()} bytes ({weight.element_size()/w_q8.element_size():.0f}× smaller)")
print(f"  mean abs error: {error8:.6f}")

# INT4 quantization (simulated — stored in int8 container)
w_q4, scale4 = quantize_tensor(weight, num_bits=4)
w_deq4 = dequantize_tensor(w_q4, scale4)
error4 = (weight - w_deq4).abs().mean().item()

print(f"\nINT4 quantized:")
print(f"  effective memory: {weight.numel() * 0.5:.0f} bytes ({weight.element_size()/0.5:.0f}× smaller)")
print(f"  mean abs error: {error4:.6f}")

print(f"\n=== Summary ===")
print(f"FP32 error: 0.000000")
print(f"INT8 error: {error8:.6f}")
print(f"INT4 error: {error4:.6f}  (higher error, much smaller footprint)")

In [ ]:
# Impact of quantization on model output
print("=== End-to-End Quantization Impact ===")

# Replace fc1 weight with dequantized INT8 weight
model_q8 = SimpleMLP(dim).to(device)
model_q8.load_state_dict(model_fp32.state_dict())

with torch.no_grad():
    # Quantize fc1 weight
    w = model_q8.fc1.weight.data.cpu()
    w_q, s = quantize_tensor(w, 8)
    model_q8.fc1.weight.data = dequantize_tensor(w_q, s).to(device)

    # TODO: Similarly quantize fc2 weight
    w2 = None  # Get fc2 weight
    w2_q, s2 = None, None  # Quantize
    model_q8.fc2.weight.data = None  # Dequantize and assign

    out_q8 = model_q8(x_test)
    out_orig = model_fp32(x_test)

output_error = (out_orig - out_q8).abs().mean().item()
relative_error = output_error / out_orig.abs().mean().item() * 100
print(f"Output mean abs error: {output_error:.6f}")
print(f"Relative error: {relative_error:.2f}%")

## Conclusions

### Technical Concepts Learned
- **MoE Architecture**: Sparse expert selection via Top-K gating, scaling capacity without scaling compute
- **Gating Network**: Router linear layer + Top-K + softmax; `scatter` for sparse gate construction
- **Load Balancing**: Auxiliary loss to prevent expert collapse during training
- **Floating-Point Formats**: FP32 (full precision) → BF16 (wide range, low precision) → FP16 (narrow range, higher precision) → FP8
- **Mixed Precision**: Using FP16/BF16 for compute with FP32 master weights for accuracy
- **Quantization**: Mapping float weights to INT8/INT4 for inference compression; trade-off between size and accuracy

### Experiment Further
- Modify `top_k` from 2 to 1 and observe the expert distribution change
- Compare per-channel vs per-tensor quantization accuracy
- Apply quantization to a real LLaMA model using `bitsandbytes` or `auto-gptq`
- Implement a capacity factor to limit maximum tokens per expert
- Benchmark MoE vs dense FFN with the same compute budget

---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
